In [ ]:
import torch
print("CUDA Available:", torch.cuda.is_available())
print("CUDA Version:", torch.version.cuda)
print("GPU Count:", torch.cuda.device_count())
print("GPU Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU Found")


In [ ]:
from unsloth import FastLanguageModel
import torch
import pandas as pd

model_name = "unsloth/DeepSeek-R1-Distill-Llama-70B-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name,  
    max_seq_length=2560,
    load_in_4bit=True        
)


FastLanguageModel.for_inference(model) 


In [ ]:
df = pd.read_csv("2021_Patient_level_prompts.csv")

df = df[(df['similarity']>0.65) & (df['similarity']<1.0)]

len(df)

In [ ]:
from tqdm import tqdm  

examples = df['instruction'].tolist()

def format_inference_prompt(instruction):

    formatted_prompt = f"""<｜begin▁of▁sentence｜><｜User｜>{instruction}<｜Assistant｜>"""

    return formatted_prompt

formatted_prompts = [format_inference_prompt(instruction) for instruction in tqdm(examples)]

tokenized_batch = tokenizer(
    formatted_prompts,
    return_tensors="pt",
    padding=True,              
).to("cuda")

In [ ]:
import re
from tqdm import tqdm

batch_size = 8  
max_new_tokens = 2048  
raw_outputs = []  
predictions = []  

tokenizer.padding_side = "left"

think_end_tag = "</think>"
answer_pattern = re.compile(r"\b(yes|no)\b", flags=re.IGNORECASE)

def extract_label(text):
    if think_end_tag not in text:
        return "Not Found"
    after_think = text.split(think_end_tag, 1)[1]
    match = answer_pattern.search(after_think)
    if match:
        return match.group(1).capitalize()
    return "Not Found"

for i in tqdm(range(0, len(tokenized_batch['input_ids']), batch_size), desc="Generating and Extracting Answers"):

    batch = {key: val[i:i+batch_size] for key, val in tokenized_batch.items()}

    output_batch = model.generate(
        **batch,
        max_length=None,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        use_cache=True,
    )

    prompt_len = batch['input_ids'].shape[1]
    new_tokens = output_batch[:, prompt_len:]

    decoded_batch = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)

    for text in decoded_batch:
        text = text.strip()
        raw_outputs.append(text)
        predictions.append(extract_label(text))

df["raw_output"] = raw_outputs
df["prediction"] = predictions

print(df["prediction"].value_counts())

In [ ]:
df.to_csv("DeepSeek-R1-70B_ZeroShot_Inference_Results.csv", index=False)